# Banka Müşteri Churn Tahmini — Modelleme

Random Forest, Gradient Boosting ve LightGBM modelleri karşılaştırılmaktadır. Sınıf dengesizliği SMOTE ile giderilmiş, en iyi model hata matrisi ile değerlendirilmiştir.

> Bu notebook `2_preprocessing.ipynb` çıktılarını (X, y) kullanmaktadır. Bağımsız çalıştırmak için ön işleme adımları aşağıya dahil edilmiştir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter

pd.set_option('display.max_columns', None)

# Veri yükleme
churn = pd.read_csv('Churn.csv')
df = churn.copy()

# Encoding
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)
df[['Geography_Germany', 'Geography_Spain', 'Gender_Male']] = df[['Geography_Germany', 'Geography_Spain', 'Gender_Male']].astype(int)

# Feature engineering
df1 = df.copy()
df1.loc[(df1['Age']>=18) & (df1['Age']<=30), 'Age'] = 1
df1.loc[(df1['Age']>30) & (df1['Age']<=40), 'Age'] = 2
df1.loc[(df1['Age']>40) & (df1['Age']<=50), 'Age'] = 3
df1.loc[(df1['Age']>50) & (df1['Age']<=60), 'Age'] = 4
df1.loc[(df1['Age']>60) & (df1['Age']<=92), 'Age'] = 5

def kredi_skor_tablosu(row):
    k = row.CreditScore
    if k < 300: return 1
    elif k < 500: return 2
    elif k < 601: return 3
    elif k < 661: return 4
    elif k < 781: return 5
    elif k < 851: return 6
    else: return 7

df1 = df1.assign(credit_score_table=df1.apply(lambda x: kredi_skor_tablosu(x), axis=1))

df1['retired'] = df['Age']
df1.loc[(df1['retired']>=65) & (df1['Geography_Germany']==1), 'retired'] = 1
df1.loc[(df1['retired']>=65) & (df1['Geography_Spain']==1), 'retired'] = 1
df1.loc[(df1['retired']>=66) & (df['Geography_Spain']==0) & (df['Geography_Germany']==0), 'retired'] = 1
df1.loc[(df1['retired']<65) & (df1['Geography_Germany']==1), 'retired'] = 0
df1.loc[(df1['retired']<65) & (df1['Geography_Spain']==1), 'retired'] = 0
df1.loc[(df1['retired']<66) & (df['Geography_Spain']==0) & (df['Geography_Germany']==0), 'retired'] = 0
df1['Tenure/NumOfProducts'] = df1['Tenure'] / df1['NumOfProducts']

df1['smallerthan405'] = df['CreditScore']
df1.loc[df1['smallerthan405'] < 405, 'smallerthan405'] = 1
df1.loc[df1['smallerthan405'] > 405, 'smallerthan405'] = 0

df1['NOP*'] = df['NumOfProducts']
df1.loc[df1['NOP*'] == 2, 'NOP*'] = 1
df1.loc[df1['NOP*'] == 1, 'NOP*'] = 2
df1.loc[df1['NOP*'] > 2, 'NOP*'] = 3

df1['Balance0'] = df1['Balance']
df1.loc[df1['Balance0'] == 0, 'Balance0'] = 0
df1.loc[df1['Balance0'] != 0, 'Balance0'] = 1

df1['ES/Age'] = df1['EstimatedSalary'] / (df['Age'] - 17)
df1['Tenure/Age'] = df1['Tenure'] / (df['Age'] - 17)
df1['Balance/ES'] = df1['Balance'] / df1['EstimatedSalary']
df1['EstimatedSalary'] = df1['EstimatedSalary'] / 12
df1['ES/Tenure'] = df1['EstimatedSalary'] / (df1['Tenure'] + 1)
df1['ES/Score'] = df1['EstimatedSalary'] / df1['credit_score_table']
df1 = df1.drop(['CreditScore', 'Tenure', 'Balance'], axis=1)

# Scaling
df1_num = df1[['Age', 'NumOfProducts', 'EstimatedSalary',
               'credit_score_table', 'Tenure/NumOfProducts', 'NOP*',
               'ES/Age', 'Tenure/Age', 'Balance/ES', 'ES/Tenure', 'ES/Score']]
col = df1_num.columns
x_transformed = pd.DataFrame(RobustScaler().fit(df1_num).transform(df1_num), columns=col)

X = pd.concat([
    x_transformed.loc[:, 'Age':'NumOfProducts'],
    df1.loc[:, 'HasCrCard':'IsActiveMember'],
    x_transformed.loc[:, 'EstimatedSalary'],
    df1.loc[:, 'Geography_Germany':'Gender_Male'],
    x_transformed.loc[:, 'credit_score_table'],
    df1.loc[:, 'retired'],
    x_transformed.loc[:, 'Tenure/NumOfProducts'],
    df1.loc[:, 'smallerthan405'],
    x_transformed.loc[:, 'NOP*'],
    df1.loc[:, 'Balance0'],
    x_transformed.loc[:, 'ES/Age':'ES/Score']
], axis=1)

y = df1['Exited']
print("Veri hazır. X shape:", X.shape)

## 4. Modelleme

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter

pd.set_option('display.max_columns', None)

In [ ]:
y = df1['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=12345)

### 4.1 Random Forest

In [ ]:
rf_model = RandomForestClassifier().fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

cv_results = cross_val_score(rf_model, X_train, y_train, cv=10, scoring='accuracy')
print('CV Doğruluk:', cv_results.mean())
print('Test Doğruluk:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
importance = rf_model.feature_importances_
plt.figure(figsize=(10, 10))
plt.barh(X.columns, importance)
plt.title('Feature Importance - Random Forest')
plt.show()

### 4.2 Gradient Boosting

In [ ]:
gbm_model = GradientBoostingClassifier().fit(X_train, y_train)
y_pred = gbm_model.predict(X_test)

cv_results = cross_val_score(gbm_model, X_train, y_train, cv=10, scoring='accuracy')
print('CV Doğruluk:', cv_results.mean())
print('Test Doğruluk:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 4.3 LightGBM

LightGBM için hiperparametre araması da yapılmış ancak varsayılan değerlerin üzerinde bir iyileştirme sağlanamamıştır.

In [ ]:
lgbm_model = LGBMClassifier().fit(X_train, y_train)
y_pred = lgbm_model.predict(X_test)

cv_results = cross_val_score(lgbm_model, X_train, y_train, cv=10, scoring='accuracy')
print('CV Doğruluk:', cv_results.mean())
print('Test Doğruluk:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 5. Sınıf Dengesizliği ve SMOTE

Veri setindeki %20/%80 dağılımı modelin azınlık sınıfını öğrenmesini zorlaştırmaktadır. SMOTE ile sentetik örnekler üretilerek sınıflar dengelenmiş, ardından modeller yeniden eğitilmiştir.

In [ ]:
smt = SMOTE(random_state=12345)
X_res, y_res = smt.fit_resample(X, y)
print('SMOTE öncesi dağılım:', Counter(y))
print('SMOTE sonrası dağılım:', Counter(y_res))

### 5.1 LightGBM + SMOTE

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.20, random_state=12345)

lgbm_model = LGBMClassifier(random_state=12345).fit(X_train, y_train)
y_pred = lgbm_model.predict(X_test)
y_train_pred = lgbm_model.predict(X_train)

cv_train = cross_val_score(lgbm_model, X_train, y_train, cv=10, scoring='accuracy')
cv_test  = cross_val_score(lgbm_model, X_test,  y_test,  cv=10, scoring='accuracy')

print('CV Doğruluk (train):', cv_train.mean())
print('CV Doğruluk (test): ', cv_test.mean())
print('Accuracy (train):   ', accuracy_score(y_train, y_train_pred))
print('Accuracy (test):    ', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 5.2 Hata Matrisi

In [ ]:
cf_matrix = confusion_matrix(y_test, y_pred)
print(cf_matrix)
sns.heatmap(cf_matrix / np.sum(cf_matrix), annot=True,
            fmt='.2%', cmap='Blues')
plt.title('Confusion Matrix - LightGBM + SMOTE')
plt.show()

## 6. Sonuç ve Değerlendirme

SMOTE uygulanmış veri seti üzerinde eğitilen Random Forest ve LightGBM modelleri birbirine yakın performans sergilemiştir.

| Model | CV Doğruluk | Train Doğruluk | Test Doğruluk |
|---|---|---|---|
| Random Forest + SMOTE | %89.24 | %92.61 | %89.39 |
| LightGBM + SMOTE | %89.47 | %92.61 | %89.45 |

En iyi sonuç LightGBM + SMOTE kombinasyonundan elde edilmiştir. 3186 tahmin üzerinden %89.42 doğruluk sağlanmış; churn olan müşteriler için precision %92, recall %87, f1-score %89 olarak gerçekleşmiştir.

Öne çıkan değişkenler `Age` ve `NumOfProducts` olmuştur. Türetilmiş değişkenlerden `Tenure/Age`, `ES/Tenure` ve `Geography_Germany` da modele kayda değer katkı sağlamıştır. `retired` ve `smallerthan405` değişkenleri feature importance açısından düşük kalmış olmakla birlikte veri setinde bırakılmıştır.